In [1]:
pip install implicit


Note: you may need to restart the kernel to use updated packages.


In [2]:
!pip install polars pyarrow -U

In [ ]:
import pandas as pd
import numpy as np
from collections import defaultdict, Counter
from tqdm import tqdm
import gc
import warnings
warnings.filterwarnings('ignore')

print("Загрузка данных")
train = pd.read_parquet('/kaggle/input/datasets/vasilisadan18/hw4-recsys/train.pq')
items = pd.read_parquet('/kaggle/input/datasets/vasilisadan18/hw4-recsys/items.pq')
test_users = pd.read_csv('/kaggle/input/datasets/vasilisadan18/hw4-recsys/test_users.csv')

train['user_id'] = train['user_id'].astype('int32')
train['item_id'] = train['item_id'].astype('int32')
train['is_purchased'] = train['is_purchased'].astype('int8')
train['rating'] = train['rating'].astype('int8')
train['timestamp'] = pd.to_datetime(train['timestamp'])
train = train.sort_values('timestamp')

print(f"Тренировочных записей: {len(train):,}")
print(f"Уникальных пользователей: {train['user_id'].nunique():,}")
print(f"Уникальных товаров: {train['item_id'].nunique():,}")
print(f"Тестовых пользователей: {len(test_users):,}")

In [ ]:
print("Разделение на train/val")
split_date = train['timestamp'].max() - pd.Timedelta(days=7)
train_data = train[train['timestamp'] < split_date].copy()
val_data = train[train['timestamp'] >= split_date].copy()

print(f"Train: {len(train_data):,} записей, {train_data['user_id'].nunique():,} пользователей")
print(f"Val: {len(val_data):,} записей, {val_data['user_id'].nunique():,} пользователей")

In [ ]:
print("Построение item-item матрицы совместных покупок")
purchase_data = train_data[train_data['is_purchased'] == 1]
user_items = purchase_data.groupby('user_id')['item_id'].apply(list)

co_counts = defaultdict(Counter)
item_users_set = defaultdict(set)

for uid, items_list in tqdm(user_items.items(), desc="Обработка покупок"):
    for i, item1 in enumerate(items_list):
        item_users_set[item1].add(uid)
        for item2 in items_list[i+1:]:
            co_counts[item1][item2] += 1
            co_counts[item2][item1] += 1
            item_users_set[item2].add(uid)

print(f"Товаров с покупками: {len(item_users_set):,}")

In [ ]:
print("Вычисление Jaccard similarity")
item_freq = Counter()
for items_list in user_items:
    item_freq.update(items_list)
top_items = [item for item, _ in item_freq.most_common(20000)]

item_sim = {}
for item1 in tqdm(top_items, desc="Jaccard"):
    users1 = item_users_set.get(item1, set())
    if len(users1) < 2:
        continue
    
    sims = []
    for item2, count in co_counts[item1].most_common(100):
        users2 = item_users_set.get(item2, set())
        union = len(users1) + len(users2) - count
        if union > 0 and count >= 2:
            jaccard = count / union
            if jaccard > 0.003:
                sims.append((item2, jaccard))
    
    if sims:
        sims.sort(key=lambda x: x[1], reverse=True)
        item_sim[item1] = sims[:50]

print(f"Похожие товары вычислены для {len(item_sim):,} товаров")

del co_counts, item_users_set, user_items
gc.collect()

In [ ]:
print("Построение последовательных паттернов покупок")
user_purchases_seq = purchase_data.groupby('user_id')['item_id'].apply(list)

next_item = defaultdict(Counter)
for items_list in tqdm(user_purchases_seq, desc="Последовательности"):
    for i in range(len(items_list) - 1):
        next_item[items_list[i]][items_list[i+1]] += 1

transitions = {}
for item_from, next_items in next_item.items():
    total = sum(next_items.values())
    if total >= 3:
        probs = [(item_to, count/total) for item_to, count in next_items.most_common(30)]
        transitions[item_from] = probs

print(f"Паттерны переходов для {len(transitions):,} товаров")
del next_item, purchase_data
gc.collect()

In [ ]:
print("Вычисление временной популярности")
item_stats = train_data.groupby('item_id').agg(
    views=('user_id', 'count'),
    purchases=('is_purchased', 'sum'),
    users=('user_id', 'nunique'),
    avg_rating=('rating', 'mean')
).reset_index()

item_stats['conversion'] = item_stats['purchases'] / item_stats['views']
item_stats['avg_rating'] = item_stats['avg_rating'].fillna(0)

train_data['week'] = train_data['timestamp'].dt.isocalendar().week.astype('int32')
max_week = train_data['week'].max()

weekly_purchases = {}
for week in range(max_week - 4, max_week + 1):
    week_data = train_data[(train_data['week'] == week) & (train_data['is_purchased'] == 1)]
    weekly_purchases[week] = week_data['item_id'].value_counts()

week_stats = pd.DataFrame(weekly_purchases).fillna(0)
week_stats['trend'] = week_stats.iloc[:, -1] - week_stats.iloc[:, 0]
week_stats['recent'] = week_stats.iloc[:, -2:].sum(axis=1)

item_stats = item_stats.merge(
    week_stats[['trend', 'recent']],
    left_on='item_id', right_index=True, how='left'
).fillna(0)

item_stats['score'] = (
    item_stats['recent'] * 3.0 +
    item_stats['purchases'] * 1.0 +
    item_stats['conversion'] * 2.0 +
    item_stats['avg_rating'] * 0.5 +
    np.maximum(item_stats['trend'], 0) * 2.0
)

top_popular = item_stats.nlargest(500, 'score')['item_id'].values
top_trending = item_stats[item_stats['trend'] > 0].nlargest(200, 'trend')['item_id'].values
top_conversion = item_stats[item_stats['views'] >= 10].nlargest(200, 'conversion')['item_id'].values

print(f"Топ товаров: {len(top_popular)} популярных, {len(top_trending)} трендовых, {len(top_conversion)} конверсионных")

In [ ]:
print("Построение истории пользователей")
user_history = {}
for uid, group in tqdm(train.groupby('user_id'), desc="Пользователи"):
    purchased = group[group['is_purchased'] == 1]
    user_history[uid] = {
        'purchased': set(purchased['item_id'].values),
        'interacted': set(group['item_id'].values),
        'last_bought': purchased.sort_values('timestamp')['item_id'].values[-10:],
        'last_seen': group.sort_values('timestamp')['item_id'].values[-10:]
    }

del train_data, train
gc.collect()

In [ ]:
def recommend(uid, n=20):
    hist = user_history.get(uid, {
        'purchased': set(),
        'interacted': set(),
        'last_bought': [],
        'last_seen': []
    })
    
    interacted = hist['interacted']
    candidates = {}
    
    items_to_match = list(hist['last_bought'][-5:]) + list(hist['last_seen'][-3:])
    for item in items_to_match:
        if item in item_sim:
            weight = 5.0 if item in hist['purchased'] else 2.0
            for sim_item, jaccard in item_sim[item]:
                if sim_item not in interacted:
                    candidates[sim_item] = candidates.get(sim_item, 0) + jaccard * weight
    
    for item in hist['last_bought'][-3:]:
        if item in transitions:
            for next_item, prob in transitions[item]:
                if next_item not in interacted and prob > 0.02:
                    candidates[next_item] = candidates.get(next_item, 0) + prob * 8.0
    
    for idx, item in enumerate(top_trending[:80]):
        if item not in interacted:
            candidates[item] = candidates.get(item, 0) + 2.0 * (1 - idx/80)
    
    for idx, item in enumerate(top_conversion[:80]):
        if item not in interacted:
            candidates[item] = candidates.get(item, 0) + 1.5 * (1 - idx/80)
    
    for idx, item in enumerate(top_popular[:200]):
        if item not in interacted:
            candidates[item] = candidates.get(item, 0) + 0.5 * (1 - idx/200)
    
    sorted_items = sorted(candidates.items(), key=lambda x: x[1], reverse=True)
    recs = []
    for item, _ in sorted_items:
        if item not in interacted and item not in recs:
            recs.append(item)
        if len(recs) >= n:
            break
    
    if len(recs) < n:
        for item in top_popular:
            if item not in interacted and item not in recs:
                recs.append(item)
            if len(recs) >= n:
                break
    
    return recs[:n]

In [ ]:
print("Локальная валидация")
val_users = list(set(val_data['user_id'].unique()) & set(user_history.keys()))[:2000]

ndcg_list = []
for uid in tqdm(val_users, desc="Валидация"):
    actual = set(val_data[(val_data['user_id'] == uid) & (val_data['is_purchased'] == 1)]['item_id'].values)
    if len(actual) == 0:
        continue
    
    recs = recommend(uid, n=20)
    
    dcg = sum((1 if item in actual else 0) / np.log2(i + 2) for i, item in enumerate(recs))
    idcg = sum(1 / np.log2(i + 2) for i in range(min(len(actual), 20)))
    ndcg = dcg / idcg if idcg > 0 else 0
    ndcg_list.append(ndcg)

print(f"NDCG@20 на валидации: {np.mean(ndcg_list):.5f}")
print(f"Оценено пользователей: {len(ndcg_list)}")

In [ ]:
print("Генерация финальных рекомендаций")
recs_list = []

for uid in tqdm(test_users['user_id'].values, desc="Рекомендации"):
    recs = recommend(uid, n=20)
    for item in recs:
        recs_list.append({'user_id': int(uid), 'item_id': int(item)})

submission = pd.DataFrame(recs_list)
submission['user_id'] = submission['user_id'].astype('int32')
submission['item_id'] = submission['item_id'].astype('int32')

assert len(submission) == len(test_users) * 20, f"Ошибка размера: {len(submission)} != {len(test_users) * 20}"
assert submission['user_id'].nunique() == len(test_users), "Ошибка количества пользователей"

submission.to_csv('submission.csv', index=False)
print(f"Сохранено: {submission.shape}")
print("\nПример:")
print(submission.head(20))

In [ ]:
from IPython.display import FileLink

display(FileLink('submission.csv'))

Score на лидерборде 0.08808

In [ ]:
import pandas as pd
import numpy as np
from collections import defaultdict, Counter
from tqdm import tqdm
import gc
import warnings
warnings.filterwarnings('ignore')

print("Загрузка данных")
train = pd.read_parquet('/kaggle/input/datasets/vasilisadan18/hw4-recsys/train.pq')
items = pd.read_parquet('/kaggle/input/datasets/vasilisadan18/hw4-recsys/items.pq')
test_users = pd.read_csv('/kaggle/input/datasets/vasilisadan18/hw4-recsys/test_users.csv')

train['user_id'] = train['user_id'].astype('int32')
train['item_id'] = train['item_id'].astype('int32')
train['is_purchased'] = train['is_purchased'].astype('int8')
train['timestamp'] = pd.to_datetime(train['timestamp'])
train = train.sort_values('timestamp')

print(f"Train: {len(train):,}, Users: {train['user_id'].nunique():,}, Items: {train['item_id'].nunique():,}")

In [ ]:
split_date = train['timestamp'].max() - pd.Timedelta(days=7)
train_data = train[train['timestamp'] < split_date].copy()
val_data = train[train['timestamp'] >= split_date].copy()
print(f"Train: {len(train_data):,}, Val: {len(val_data):,}")

In [ ]:
print("Item-item совместные покупки")
purchase_data = train_data[train_data['is_purchased'] == 1]
user_items = purchase_data.groupby('user_id')['item_id'].apply(list)

co_counts = defaultdict(Counter)
item_users_set = defaultdict(set)

for uid, items_list in tqdm(user_items.items(), desc="Обработка"):
    for i, item1 in enumerate(items_list):
        item_users_set[item1].add(uid)
        for item2 in items_list[i+1:]:
            co_counts[item1][item2] += 1
            co_counts[item2][item1] += 1
            item_users_set[item2].add(uid)

item_freq = Counter()
for items_list in user_items:
    item_freq.update(items_list)
top_items = [item for item, _ in item_freq.most_common(15000)]

item_sim = {}
for item1 in tqdm(top_items, desc="Jaccard"):
    users1 = item_users_set.get(item1, set())
    if len(users1) < 2:
        continue
    sims = []
    for item2, count in co_counts[item1].most_common(100):
        users2 = item_users_set.get(item2, set())
        union = len(users1) + len(users2) - count
        if union > 0 and count >= 2:
            jaccard = count / union
            if jaccard > 0.003:
                sims.append((item2, jaccard))
    if sims:
        sims.sort(key=lambda x: x[1], reverse=True)
        item_sim[item1] = sims[:50]

print(f"Похожие товары: {len(item_sim)}")
del co_counts, item_users_set, user_items
gc.collect()

In [ ]:
print("Последовательные покупки")
user_purchases_seq = purchase_data.groupby('user_id')['item_id'].apply(list)
next_item = defaultdict(Counter)

for items_list in tqdm(user_purchases_seq, desc="Sequences"):
    for i in range(len(items_list) - 1):
        next_item[items_list[i]][items_list[i+1]] += 1

transitions = {}
for item_from, next_items in next_item.items():
    total = sum(next_items.values())
    if total >= 3:
        probs = [(item_to, count/total) for item_to, count in next_items.most_common(30)]
        transitions[item_from] = probs

print(f"Переходов: {len(transitions)}")
del next_item, purchase_data, user_purchases_seq
gc.collect()

In [ ]:
print("Популярность и тренды")
item_stats = train_data.groupby('item_id').agg(
    views=('user_id', 'count'),
    purchases=('is_purchased', 'sum'),
    users=('user_id', 'nunique'),
    avg_rating=('rating', 'mean')
).reset_index()

item_stats['conversion'] = item_stats['purchases'] / item_stats['views']
item_stats['avg_rating'] = item_stats['avg_rating'].fillna(0)

train_data['week'] = train_data['timestamp'].dt.isocalendar().week.astype('int32')
max_week = train_data['week'].max()

weekly_purchases = {}
for week in range(max_week - 4, max_week + 1):
    week_data = train_data[(train_data['week'] == week) & (train_data['is_purchased'] == 1)]
    weekly_purchases[week] = week_data['item_id'].value_counts()

week_stats = pd.DataFrame(weekly_purchases).fillna(0)
week_stats['trend'] = week_stats.iloc[:, -1] - week_stats.iloc[:, 0]
week_stats['recent'] = week_stats.iloc[:, -2:].sum(axis=1)

item_stats = item_stats.merge(week_stats[['trend', 'recent']], left_on='item_id', right_index=True, how='left').fillna(0)

item_stats['score'] = (
    item_stats['recent'] * 3.0 +
    item_stats['purchases'] * 1.0 +
    item_stats['conversion'] * 2.0 +
    item_stats['avg_rating'] * 0.5 +
    np.maximum(item_stats['trend'], 0) * 2.0
)

top_popular = item_stats.nlargest(500, 'score')['item_id'].values
top_trending = item_stats[item_stats['trend'] > 0].nlargest(200, 'trend')['item_id'].values
top_conversion = item_stats[item_stats['views'] >= 10].nlargest(200, 'conversion')['item_id'].values

print(f"Топ товаров: {len(top_popular)} популярных")

In [ ]:
print("Повторные покупки")
val_purchases = val_data[val_data['is_purchased'] == 1]
val_users_purch = val_purchases['user_id'].unique()

item_rebuy = defaultdict(lambda: {'total': 0, 'rebought': 0})
for uid in val_users_purch:
    hist_purchased = set(train_data[(train_data['user_id'] == uid) & (train_data['is_purchased'] == 1)]['item_id'].values)
    val_purchased = set(val_data[(val_data['user_id'] == uid) & (val_data['is_purchased'] == 1)]['item_id'].values)
    for item in val_purchased:
        item_rebuy[item]['total'] += 1
        if item in hist_purchased:
            item_rebuy[item]['rebought'] += 1

rebuy_items = {}
for item, stats in item_rebuy.items():
    if stats['total'] >= 3:
        rate = stats['rebought'] / stats['total']
        if rate > 0.15:
            rebuy_items[item] = rate

print(f"Товаров с повторными покупками: {len(rebuy_items)}")

In [ ]:
print("История пользователей")
user_history_train = {}
for uid, group in tqdm(train_data.groupby('user_id'), desc="История"):
    purchased = group[group['is_purchased'] == 1]
    user_history_train[uid] = {
        'purchased': set(purchased['item_id'].values),
        'interacted': set(group['item_id'].values),
        'last_bought': purchased.sort_values('timestamp')['item_id'].values[-10:],
        'last_seen': group.sort_values('timestamp')['item_id'].values[-10:]
    }

del train_data
gc.collect()

In [ ]:
def recommend(uid, n=20):
    hist = user_history_train.get(uid, {
        'purchased': set(), 'interacted': set(),
        'last_bought': [], 'last_seen': []
    })
    
    interacted = hist['interacted']
    candidates = {}
    
    for idx, item in enumerate(top_popular[:300]):
        if item not in interacted:
            candidates[item] = 10.0 * (1 - idx/300)

    for idx, item in enumerate(top_trending[:100]):
        if item not in interacted:
            candidates[item] = candidates.get(item, 0) + 5.0 * (1 - idx/100)

    for idx, item in enumerate(top_conversion[:100]):
        if item not in interacted:
            candidates[item] = candidates.get(item, 0) + 4.0 * (1 - idx/100)
    
    for item in list(hist['purchased'])[:10]:
        if item in rebuy_items:
            candidates[item] = candidates.get(item, 0) + rebuy_items[item] * 20.0
    for item in list(hist['last_bought'][-7:]) + list(hist['last_seen'][-5:]):
        if item in item_sim:
            weight = 8.0 if item in hist['purchased'] else 4.0
            for sim_item, jaccard in item_sim[item]:
                if sim_item not in interacted:
                    candidates[sim_item] = candidates.get(sim_item, 0) + jaccard * weight
    
    for item in hist['last_bought'][-5:]:
        if item in transitions:
            for next_item, prob in transitions[item]:
                if next_item not in interacted and prob > 0.01:
                    candidates[next_item] = candidates.get(next_item, 0) + prob * 12.0
    
    sorted_items = sorted(candidates.items(), key=lambda x: x[1], reverse=True)
    recs = []
    for item, _ in sorted_items:
        if item not in interacted and item not in recs:
            recs.append(item)
        if len(recs) >= n:
            break
    
    if len(recs) < n:
        for item in top_popular:
            if item not in interacted and item not in recs:
                recs.append(item)
            if len(recs) >= n:
                break
    
    return recs[:n]

In [ ]:
print("Валидация")
common_val = list(set(val_data[val_data['is_purchased'] == 1]['user_id'].unique()) & set(user_history_train.keys()))
print(f"Пользователей: {len(common_val)}")

ndcg_list = []
for uid in tqdm(common_val[:2000], desc="Валидация"):
    actual = set(val_data[(val_data['user_id'] == uid) & (val_data['is_purchased'] == 1)]['item_id'].values)
    recs = recommend(uid, n=20)
    dcg = sum((1 if item in actual else 0) / np.log2(i + 2) for i, item in enumerate(recs))
    idcg = sum(1 / np.log2(i + 2) for i in range(min(len(actual), 20)))
    ndcg = dcg / idcg if idcg > 0 else 0
    ndcg_list.append(ndcg)

print(f"NDCG@20: {np.mean(ndcg_list):.5f}")
print(f"Медиана: {np.median(ndcg_list):.5f}")
print(f"Доля нулей: {(np.array(ndcg_list) == 0).mean():.3f}")

In [ ]:
print("Генерация сабмита")
recs_list = []

for uid in tqdm(test_users['user_id'].values, desc="Рекомендации"):
    recs = recommend(uid, n=20)
    for item in recs:
        recs_list.append({'user_id': int(uid), 'item_id': int(item)})

submission = pd.DataFrame(recs_list)
submission['user_id'] = submission['user_id'].astype('int32')
submission['item_id'] = submission['item_id'].astype('int32')
submission.to_csv('submission2.csv', index=False)

In [ ]:
from IPython.display import FileLink

display(FileLink('submission2.csv'))

In [ ]:
скор на лидерборде 0.07996

In [ ]:
import pandas as pd
import numpy as np
from collections import defaultdict, Counter
from tqdm import tqdm
from catboost import CatBoostRanker, Pool
import gc
import warnings
warnings.filterwarnings('ignore')

print("Загрузка данных")
train = pd.read_parquet('/kaggle/input/datasets/vasilisadan18/hw4-recsys/train.pq')
items = pd.read_parquet('/kaggle/input/datasets/vasilisadan18/hw4-recsys/items.pq')
test_users = pd.read_csv('/kaggle/input/datasets/vasilisadan18/hw4-recsys/test_users.csv')

train['user_id'] = train['user_id'].astype('int32')
train['item_id'] = train['item_id'].astype('int32')
train['is_purchased'] = train['is_purchased'].astype('int8')
train['rating'] = train['rating'].astype('int8')
train['timestamp'] = pd.to_datetime(train['timestamp'])
train = train.sort_values('timestamp')

split_date = train['timestamp'].max() - pd.Timedelta(days=7)
train_data = train[train['timestamp'] < split_date].copy()
val_data = train[train['timestamp'] >= split_date].copy()

print(f"Train: {len(train_data):,}, Val: {len(val_data):,}")



In [ ]:
item_stats = train_data.groupby('item_id').agg(
    item_views=('user_id', 'count'),
    item_purchases=('is_purchased', 'sum'),
    item_avg_rating=('rating', 'mean')
).reset_index()
item_stats['item_purchase_rate'] = item_stats['item_purchases'] / item_stats['item_views']
item_stats['item_avg_rating'] = item_stats['item_avg_rating'].fillna(0)


user_stats = train_data.groupby('user_id').agg(
    user_views=('item_id', 'count'),
    user_purchases=('is_purchased', 'sum'),
    user_avg_rating=('rating', 'mean')
).reset_index()
user_stats['user_conversion'] = user_stats['user_purchases'] / user_stats['user_views']
user_stats['user_avg_rating'] = user_stats['user_avg_rating'].fillna(0)

item_stats['popularity'] = item_stats['item_purchases'] * 0.6 + item_stats['item_purchase_rate'] * 0.4
top_popular = item_stats.nlargest(1000, 'popularity')['item_id'].values



In [ ]:
print("Item-Item сходство")
purchases = train_data[train_data['is_purchased'] == 1]
user_items = purchases.groupby('user_id')['item_id'].apply(list)

item_users = defaultdict(set)
for uid, items_list in user_items.items():
    for item in items_list:
        item_users[item].add(uid)

top_items = [item for item, _ in Counter(i for items in user_items for i in items).most_common(15000)]

item_sim_dict = {}
for item1 in tqdm(top_items, desc="Jaccard"):
    users1 = item_users.get(item1, set())
    if len(users1) < 2:
        continue
    sims = Counter()
    for uid in users1:
        for item2 in user_items[uid]:
            if item2 != item1:
                sims[item2] += 1
    jaccards = {}
    for item2, inter in sims.items():
        users2 = item_users.get(item2, set())
        union = len(users1) + len(users2) - inter
        if union > 0 and inter >= 2:
            j = inter / union
            if j > 0.003:
                jaccards[item2] = j
    if jaccards:
        item_sim_dict[item1] = jaccards

print(f"Сходство для {len(item_sim_dict)} товаров")
del user_items, item_users

print("Последовательные переходы")
user_seqs = purchases.groupby('user_id')['item_id'].apply(list)
next_item = defaultdict(Counter)
for seq in user_seqs:
    for i in range(len(seq)-1):
        next_item[seq[i]][seq[i+1]] += 1
transition_dict = {}
for item_from, nexts in next_item.items():
    total = sum(nexts.values())
    if total >= 3:
        transition_dict[item_from] = {k: v/total for k, v in nexts.most_common(30)}
print(f"Переходов: {len(transition_dict)}")
del user_seqs, next_item


In [ ]:
# История пользователей
print("История пользователей")
user_history = {}
for uid, group in train_data.groupby('user_id'):
    purchased = group[group['is_purchased'] == 1]
    user_history[uid] = {
        'purchased': set(purchased['item_id'].values),
        'interacted': set(group['item_id'].values),
        'last_bought': list(purchased.sort_values('timestamp')['item_id'].values[-5:]),
        'last_seen': list(group.sort_values('timestamp')['item_id'].values[-5:])
    }



In [ ]:
# Генерация кандидатов 
def get_candidates_fast(uid):
    hist = user_history.get(uid, {'purchased': set(), 'interacted': set(), 'last_bought': [], 'last_seen': []})
    candidates = set()
    for item in hist['last_bought'][-5:] + hist['last_seen'][-3:]:
        if item in item_sim_dict:
            candidates.update(list(item_sim_dict[item].keys())[:20])
    for item in hist['last_bought'][-3:]:
        if item in transition_dict:
            candidates.update(list(transition_dict[item].keys())[:15])
    candidates.update(hist['purchased'])
    for item in top_popular[:80]:
        candidates.add(item)
    return [c for c in candidates if c not in hist['interacted']][:150]


In [ ]:
#  Подготовка обучающего набора для CatBoost
print("Подготовка данных для обучения")

# Выбираем пользователей из val, у которых есть история
val_users = val_data['user_id'].unique()
val_users_with_hist = [u for u in val_users if u in user_history]

# Собираем кандидатов и таргеты для каждого пользователя
train_rows = []
for uid in tqdm(val_users_with_hist[:10000], desc="Train users"):
    candidates = get_candidates_fast(uid)
    if not candidates:
        continue

    val_user_data = val_data[val_data['user_id'] == uid]
    purchased_set = set(val_user_data[val_user_data['is_purchased'] == 1]['item_id'])
    for iid in candidates:
        train_rows.append({'user_id': uid, 'item_id': iid, 'target': 1 if iid in purchased_set else 0})

    for iid in purchased_set:
        if iid not in candidates:
            train_rows.append({'user_id': uid, 'item_id': iid, 'target': 1})

train_df = pd.DataFrame(train_rows)
print(f"Обучающих примеров: {len(train_df)}")

In [ ]:
# признаки для пар user-item
print("Вычисление признаков")
user_stats_indexed = user_stats.set_index('user_id')
item_stats_indexed = item_stats.set_index('item_id')

train_df['user_views'] = train_df['user_id'].map(user_stats_indexed['user_views']).fillna(0)
train_df['user_purchases'] = train_df['user_id'].map(user_stats_indexed['user_purchases']).fillna(0)
train_df['user_conversion'] = train_df['user_id'].map(user_stats_indexed['user_conversion']).fillna(0)
train_df['user_avg_rating'] = train_df['user_id'].map(user_stats_indexed['user_avg_rating']).fillna(0)

train_df['item_views'] = train_df['item_id'].map(item_stats_indexed['item_views']).fillna(0)
train_df['item_purchases'] = train_df['item_id'].map(item_stats_indexed['item_purchases']).fillna(0)
train_df['item_purchase_rate'] = train_df['item_id'].map(item_stats_indexed['item_purchase_rate']).fillna(0)
train_df['item_avg_rating'] = train_df['item_id'].map(item_stats_indexed['item_avg_rating']).fillna(0)

print("Расчет Jaccard и transition для train")

# Сгруппируем train_df по пользователям и вычислим признаки
def calc_interaction_features(group):
    uid = group.name
    hist = user_history.get(uid, {'last_bought': [], 'purchased': set()})
    last_bought = hist['last_bought'][-5:]
    
    # Max jaccard
    jaccard_vals = []
    for iid in group['item_id']:
        max_j = 0.0
        for item in last_bought:
            sims = item_sim_dict.get(item, {})
            j = sims.get(iid, 0.0)
            if j > max_j: max_j = j
        jaccard_vals.append(max_j)
    
    # Max transition
    trans_vals = []
    for iid in group['item_id']:
        max_t = 0.0
        for item in last_bought:
            trans = transition_dict.get(item, {})
            t = trans.get(iid, 0.0)
            if t > max_t: max_t = t
        trans_vals.append(max_t)
    
    purchased_before = [1 if iid in hist['purchased'] else 0 for iid in group['item_id']]
    
    return pd.DataFrame({
        'user_id': uid,
        'item_id': group['item_id'].values,
        'max_jaccard': jaccard_vals,
        'max_transition': trans_vals,
        'was_purchased_before': purchased_before
    })

# Применяем группировку
interaction_features = train_df.groupby('user_id').apply(calc_interaction_features).reset_index(drop=True)
train_df = train_df.merge(interaction_features, on=['user_id', 'item_id'], how='left')

# Категориальные совпадения
items['n_categories'] = items['category_tags'].apply(lambda x: len(x) if isinstance(x, list) else 0)
items['n_authors'] = items['author_ids'].apply(lambda x: len(x) if isinstance(x, list) else 0)
item_cats_set = dict(zip(items['item_id'], items['category_tags'].apply(lambda x: set(x) if isinstance(x, list) else set())))
item_auths_set = dict(zip(items['item_id'], items['author_ids'].apply(lambda x: set(x) if isinstance(x, list) else set())))

# Топ категории/авторы для пользователей
user_top_cats = {}
user_top_auths = {}
purch_with_items = train_data[train_data['is_purchased'] == 1][['user_id','item_id']].merge(
    items[['item_id','category_tags','author_ids']], on='item_id', how='left')
for uid, group in purch_with_items.groupby('user_id'):
    cat_counter = Counter()
    auth_counter = Counter()
    for _, row in group.iterrows():
        if isinstance(row['category_tags'], list):
            cat_counter.update(row['category_tags'])
        if isinstance(row['author_ids'], list):
            auth_counter.update(row['author_ids'])
    user_top_cats[uid] = {c for c, _ in cat_counter.most_common(10)}
    user_top_auths[uid] = {a for a, _ in auth_counter.most_common(10)}

def compute_cat_match(row):
    uid = row['user_id']
    iid = row['item_id']
    user_cats = user_top_cats.get(uid, set())
    item_cats = item_cats_set.get(iid, set())
    if item_cats:
        return len(user_cats & item_cats) / len(item_cats)
    return 0.0

def compute_auth_match(row):
    uid = row['user_id']
    iid = row['item_id']
    user_auths = user_top_auths.get(uid, set())
    item_auths = item_auths_set.get(iid, set())
    if item_auths:
        return len(user_auths & item_auths) / len(item_auths)
    return 0.0

train_df['category_match'] = train_df.apply(compute_cat_match, axis=1)
train_df['author_match'] = train_df.apply(compute_auth_match, axis=1)
train_df['item_n_categories'] = train_df['item_id'].map(items.set_index('item_id')['n_categories']).fillna(0)
train_df['item_n_authors'] = train_df['item_id'].map(items.set_index('item_id')['n_authors']).fillna(0)

# Определяем признаки
feature_cols = [
    'user_views', 'user_purchases', 'user_conversion', 'user_avg_rating',
    'item_views', 'item_purchases', 'item_purchase_rate', 'item_avg_rating',
    'max_jaccard', 'max_transition', 'was_purchased_before',
    'category_match', 'author_match', 'item_n_categories', 'item_n_authors'
]

In [ ]:
# Обучение CatBoostRanker
print("\nОбучение CatBoost")
train_df = train_df.sort_values('user_id')
train_pool = Pool(
    data=train_df[feature_cols],
    label=train_df['target'],
    group_id=train_df['user_id']
)

model = CatBoostRanker(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    loss_function='YetiRank',
    custom_metric='NDCG:top=20',
    random_seed=42,
    verbose=50
)
model.fit(train_pool, early_stopping_rounds=50)
print(f"Лучшая итерация: {model.get_best_iteration()}")

In [ ]:
# Генерация сабмита
print("\nГенерация сабмита")
test_user_ids = test_users['user_id'].values
batch_size = 2000
all_recs = []

for start in tqdm(range(0, len(test_user_ids), batch_size), desc="Test batches"):
    batch_uids = test_user_ids[start:start+batch_size]
    # Собираем кандидатов
    batch_data = []
    for uid in batch_uids:
        cands = get_candidates_fast(uid)
        if not cands:
            cands = list(top_popular[:100])
        for iid in cands:
            batch_data.append({'user_id': uid, 'item_id': iid})
    if not batch_data:
        continue
    batch_df = pd.DataFrame(batch_data)
    
    # Признаки
    batch_df['user_views'] = batch_df['user_id'].map(user_stats_indexed['user_views']).fillna(0)
    batch_df['user_purchases'] = batch_df['user_id'].map(user_stats_indexed['user_purchases']).fillna(0)
    batch_df['user_conversion'] = batch_df['user_id'].map(user_stats_indexed['user_conversion']).fillna(0)
    batch_df['user_avg_rating'] = batch_df['user_id'].map(user_stats_indexed['user_avg_rating']).fillna(0)
    batch_df['item_views'] = batch_df['item_id'].map(item_stats_indexed['item_views']).fillna(0)
    batch_df['item_purchases'] = batch_df['item_id'].map(item_stats_indexed['item_purchases']).fillna(0)
    batch_df['item_purchase_rate'] = batch_df['item_id'].map(item_stats_indexed['item_purchase_rate']).fillna(0)
    batch_df['item_avg_rating'] = batch_df['item_id'].map(item_stats_indexed['item_avg_rating']).fillna(0)
    
    # Взаимодействия
    inter_feats = batch_df.groupby('user_id').apply(calc_interaction_features).reset_index(drop=True)
    batch_df = batch_df.merge(inter_feats, on=['user_id','item_id'], how='left')
    
    # Категориальные совпадения
    batch_df['category_match'] = batch_df.apply(compute_cat_match, axis=1)
    batch_df['author_match'] = batch_df.apply(compute_auth_match, axis=1)
    batch_df['item_n_categories'] = batch_df['item_id'].map(items.set_index('item_id')['n_categories']).fillna(0)
    batch_df['item_n_authors'] = batch_df['item_id'].map(items.set_index('item_id')['n_authors']).fillna(0)
    
    # Предсказание
    X_batch = batch_df[feature_cols].fillna(0)
    scores = model.predict(X_batch)
    batch_df['score'] = scores
    
    # Выбор топ-20 для каждого пользователя
    for uid, group in batch_df.groupby('user_id'):
        group = group.sort_values('score', ascending=False)
        interacted = user_history[uid]['interacted'] if uid in user_history else set()
        recs = []
        for _, row in group.iterrows():
            if row['item_id'] not in interacted and row['item_id'] not in recs:
                recs.append(row['item_id'])
            if len(recs) >= 20:
                break
        while len(recs) < 20:
            for item in top_popular:
                if item not in interacted and item not in recs:
                    recs.append(item)
                if len(recs) >= 20:
                    break
        for r in recs[:20]:
            all_recs.append({'user_id': int(uid), 'item_id': int(r)})

submission = pd.DataFrame(all_recs)
submission['user_id'] = submission['user_id'].astype('int32')
submission['item_id'] = submission['item_id'].astype('int32')
submission.to_csv('submission4.csv', index=False)
print(f"Сохранено: {submission.shape}")

In [ ]:
from IPython.display import FileLink

display(FileLink('submission4.csv'))

скор ниже 0.06448

In [38]:
# Импорты и загрузка данных
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from implicit import als
import lightgbm as lgb
from collections import defaultdict
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

print("Загрузка данных...")

DATA_DIR = "/kaggle/input/datasets/vasilisadan18/hw4-recsys"
TOP_K = 20
RANDOM_SEED = 42

train = pd.read_parquet(f"{DATA_DIR}/train.pq")
items = pd.read_parquet(f"{DATA_DIR}/items.pq")
test_users = pd.read_csv(f"{DATA_DIR}/test_users.csv")

train["timestamp"] = pd.to_datetime(train["timestamp"])
train = train.sort_values("timestamp")

print(f"Тренировочных взаимодействий: {len(train):,}")
print(f"Уникальных пользователей: {train['user_id'].nunique():,}")
print(f"Уникальных товаров: {train['item_id'].nunique():,}")
print(f"Тестовых пользователей: {len(test_users):,}")

Загрузка данных...
Тренировочных взаимодействий: 11,373,426
Уникальных пользователей: 348,111
Уникальных товаров: 31,221
Тестовых пользователей: 185,282


In [39]:
# Разделение на train/val по времени
print("Разделение на тренировочную и валидационную выборки")

cutoff = train["timestamp"].quantile(0.85)
val_data = train[train["timestamp"] >= cutoff].copy()
train_data = train[train["timestamp"] < cutoff].copy()

print(f"Тренировочных записей: {len(train_data):,}")
print(f"Валидационных записей: {len(val_data):,}")
print(f"Дата разделения: {cutoff}")

Разделение на тренировочную и валидационную выборки...
Тренировочных записей: 9,667,412
Валидационных записей: 1,706,014
Дата разделения: 2016-07-08 01:10:38.250000


In [40]:
#  Вычисление весов взаимодействий
print("Вычисление весов взаимодействий")
def compute_weights(df):
    """Вычисляет веса взаимодействий на основе покупок, рейтинга и давности"""
    df = df.copy()
    df["weight"] = 1.0
    
    # Покупки важнее просмотров
    if "is_purchased" in df.columns:
        df.loc[df["is_purchased"] == True, "weight"] = 5.0
    
    # Высокий рейтинг увеличивает вес
    if "rating" in df.columns:
        mask = df["rating"] > 0
        df.loc[mask, "weight"] += df.loc[mask, "rating"] / 5.0 * 2.0
    
    # Давность взаимодействия (более новые важнее)
    ts_min = df["timestamp"].min()
    ts_max = df["timestamp"].max()
    ts_range = (ts_max - ts_min).total_seconds()
    if ts_range > 0:
        df["recency"] = (df["timestamp"] - ts_min).dt.total_seconds() / ts_range
        df["weight"] *= (0.5 + 0.5 * df["recency"])
    
    return df.groupby(["user_id", "item_id"])["weight"].max().reset_index()

train_agg = compute_weights(train_data)
print(f"Взвешенных взаимодействий: {len(train_agg):,}")
print(f"Средний вес: {train_agg['weight'].mean():.2f}")

Вычисление весов взаимодействий...
Взвешенных взаимодействий: 9,667,412
Средний вес: 2.39


In [41]:
#  Создание индексных отображений
print("Создание индексных отображений")

all_item_ids = train_agg["item_id"].unique()
all_user_ids = train_agg["user_id"].unique()

user2idx = {u: i for i, u in enumerate(all_user_ids)}
item2idx = {it: i for i, it in enumerate(all_item_ids)}
idx2item = {i: it for it, i in item2idx.items()}

train_agg["uidx"] = train_agg["user_id"].map(user2idx)
train_agg["iidx"] = train_agg["item_id"].map(item2idx)

n_users = len(all_user_ids)
n_items = len(all_item_ids)

print(f"Пользователей в индексе: {n_users:,}")
print(f"Товаров в индексе: {n_items:,}")

Создание индексных отображений...
Пользователей в индексе: 325,261
Товаров в индексе: 30,008


In [23]:
# Создание разреженной матрицы и обучение iALS

print("Создание разреженной матрицы и обучение iALS")

ALS_ALPHA = 40
sparse_item_user = csr_matrix(
    (train_agg["weight"].values * ALS_ALPHA,
     (train_agg["iidx"].values, train_agg["uidx"].values)),
    shape=(n_items, n_users)
)
sparse_user_item = sparse_item_user.T.tocsr()

print(f"Размер матрицы: {sparse_item_user.shape}")
print(f"Ненулевых элементов: {sparse_item_user.nnz:,}")

als_model = als.AlternatingLeastSquares(
    factors=128,
    iterations=25,
    regularization=0.01,
    random_state=RANDOM_SEED,
    use_gpu=False,
)
als_model.fit(sparse_item_user)
print("iALS модель обучена!")

Создание разреженной матрицы и обучение iALS...
Размер матрицы: (30008, 325261)
Ненулевых элементов: 9,667,412


  0%|          | 0/25 [00:00<?, ?it/s]

iALS модель обучена!


In [42]:
# Признаки товаров и пользователей

print("Построение признаков товаров и пользователей")

# Признаки товаров из каталога
item_feat = {}
for _, row in items.iterrows():
    iid = row["item_id"]
    tags = set(row["category_tags"]) if isinstance(row.get("category_tags"), list) else set()
    series = set(row["series_id"]) if isinstance(row.get("series_id"), list) else set()
    authors = set(row["author_ids"]) if isinstance(row.get("author_ids"), list) else set()
    item_feat[iid] = {"tags": tags, "series": series, "authors": authors}

# Статистики товаров
item_stats = train_data.groupby("item_id").agg(
    item_interactions=("user_id", "count"),
    item_purchases=("is_purchased", "sum") if "is_purchased" in train_data.columns else ("user_id", "count"),
    item_unique_users=("user_id", "nunique"),
).reset_index()

if "rating" in train_data.columns:
    item_rating_stats = train_data[train_data["rating"] > 0].groupby("item_id").agg(
        item_avg_rating=("rating", "mean"),
        item_rating_count=("rating", "count"),
    ).reset_index()
    item_stats = item_stats.merge(item_rating_stats, on="item_id", how="left")
    item_stats["item_avg_rating"].fillna(0, inplace=True)
    item_stats["item_rating_count"].fillna(0, inplace=True)

item_stats["item_popularity_rank"] = item_stats["item_interactions"].rank(ascending=False)

# Статистики пользователей
user_stats = train_data.groupby("user_id").agg(
    user_interactions=("item_id", "count"),
    user_unique_items=("item_id", "nunique"),
).reset_index()

if "is_purchased" in train_data.columns:
    user_purchase_stats = train_data.groupby("user_id")["is_purchased"].agg(
        user_purchase_rate="mean"
    ).reset_index()
    user_stats = user_stats.merge(user_purchase_stats, on="user_id", how="left")

print(f"Признаков товаров: {item_stats.shape[1]}")
print(f"Признаков пользователей: {user_stats.shape[1]}")

Построение признаков товаров и пользователей...
Признаков товаров: 7
Признаков пользователей: 4


In [25]:
# Функции для генерации кандидатов
print("Подготовка функций генерации кандидатов.")

CANDIDATES_PER_USER = 100

user_seen = train_agg.groupby("user_id")["item_id"].apply(set).to_dict()

popularity_fallback = item_stats.sort_values("item_interactions", ascending=False)["item_id"].tolist()

def get_candidates_for_user(user_id, n=CANDIDATES_PER_USER):
    candidates = []
    
    if user_id in user2idx:
        uidx = user2idx[user_id]
        user_vec = sparse_user_item[uidx]
        rec_idxs, rec_scores = als_model.recommend(
            uidx, user_vec, N=n // 2, filter_already_liked_items=True
        )
        for idx, score in zip(rec_idxs, rec_scores):
            candidates.append((idx2item[idx], score, "als"))
    
    if user_id in user_seen:
        seen_items = list(user_seen[user_id])
        recent_items = seen_items[-5:]
        seen_set = set(seen_items)
        
        knn_candidates = []
        for seed_item in recent_items:
            if seed_item not in item2idx:
                continue
            seed_iidx = item2idx[seed_item]
            seed_vec = als_model.item_factors[seed_iidx].reshape(1, -1)
            sims = als_model.item_factors @ seed_vec.T
            sims = sims.flatten()
            top_idxs = np.argsort(sims)[::-1][1:20]
            for idx in top_idxs:
                cand_item = idx2item[idx]
                if cand_item not in seen_set:
                    knn_candidates.append((cand_item, float(sims[idx]), "knn"))
        
        candidates.extend(knn_candidates[:n // 4])
    
    seen_set = user_seen.get(user_id, set())
    pop_count = 0
    for pop_item in popularity_fallback:
        if pop_item not in seen_set and pop_item not in {c[0] for c in candidates}:
            candidates.append((pop_item, 0.0, "pop"))
            pop_count += 1
            if pop_count >= n // 4:
                break
    
    seen_cands = set()
    unique_candidates = []
    for c in candidates:
        if c[0] not in seen_cands:
            seen_cands.add(c[0])
            unique_candidates.append(c)
    
    return unique_candidates[:n]


Подготовка функций генерации кандидатов...


In [26]:
# Функции для построения признаков

print("Подготовка функций вычисления признаков")

def build_user_profile(user_id, history_df):
    user_hist = history_df[history_df["user_id"] == user_id]
    
    tag_counts = defaultdict(float)
    author_counts = defaultdict(float)
    series_counts = defaultdict(float)
    
    for _, row in user_hist.iterrows():
        iid = row["item_id"]
        w = row.get("weight", 1.0)
        feat = item_feat.get(iid, {})
        for t in feat.get("tags", set()):
            tag_counts[t] += w
        for a in feat.get("authors", set()):
            author_counts[a] += w
        for s in feat.get("series", set()):
            series_counts[s] += w
    
    return tag_counts, author_counts, series_counts


def compute_features(user_id, item_id, als_score, source,
                     tag_profile, author_profile, series_profile):
    feat = item_feat.get(item_id, {})
    
    tag_score = sum(tag_profile.get(t, 0) for t in feat.get("tags", set()))
    author_score = sum(author_profile.get(a, 0) for a in feat.get("authors", set()))
    series_score = sum(series_profile.get(s, 0) for s in feat.get("series", set()))
    
    istats = item_stats[item_stats["item_id"] == item_id]
    i_interactions = float(istats["item_interactions"].values[0]) if len(istats) > 0 else 0
    i_unique_users = float(istats["item_unique_users"].values[0]) if len(istats) > 0 else 0
    i_pop_rank = float(istats["item_popularity_rank"].values[0]) if len(istats) > 0 else 99999
    i_avg_rating = float(istats["item_avg_rating"].values[0]) if (len(istats) > 0 and "item_avg_rating" in istats.columns) else 0
    
    ustats = user_stats[user_stats["user_id"] == user_id]
    u_interactions = float(ustats["user_interactions"].values[0]) if len(ustats) > 0 else 0
    u_unique_items = float(ustats["user_unique_items"].values[0]) if len(ustats) > 0 else 0
    u_purchase_rate = float(ustats["user_purchase_rate"].values[0]) if (len(ustats) > 0 and "user_purchase_rate" in ustats.columns) else 0
    
    source_als = 1.0 if source == "als" else 0.0
    source_knn = 1.0 if source == "knn" else 0.0
    source_pop = 1.0 if source == "pop" else 0.0
    
    return {
        "als_score": als_score,
        "tag_score": tag_score,
        "author_score": author_score,
        "series_score": series_score,
        "n_tags": len(feat.get("tags", set())),
        "n_authors": len(feat.get("authors", set())),
        "n_series": len(feat.get("series", set())),
        "item_interactions": i_interactions,
        "item_unique_users": i_unique_users,
        "item_pop_rank": i_pop_rank,
        "item_avg_rating": i_avg_rating,
        "user_interactions": u_interactions,
        "user_unique_items": u_unique_items,
        "user_purchase_rate": u_purchase_rate,
        "source_als": source_als,
        "source_knn": source_knn,
        "source_pop": source_pop,
        "tag_score_norm": tag_score / (u_interactions + 1),
        "author_score_norm": author_score / (u_interactions + 1),
        "series_score_norm": series_score / (u_interactions + 1),
    }

print("Функции вычисления признаков готовы!")

Подготовка функций вычисления признаков...
Функции вычисления признаков готовы!


In [28]:
# Построение обучающих данных для LightGBM

print("Построение обучающих данных для LightGBM")

val_user_ids = val_data["user_id"].unique()
val_users_in_train = [u for u in val_user_ids if u in user2idx]


MAX_TRAIN_USERS = 5000
if len(val_users_in_train) > MAX_TRAIN_USERS:
    np.random.seed(RANDOM_SEED)
    val_users_in_train = np.random.choice(val_users_in_train, MAX_TRAIN_USERS, replace=False)

print(f"Пользователей для обучения LightGBM: {len(val_users_in_train)}")

val_relevant = (
    val_data[val_data["user_id"].isin(val_users_in_train)]
    .groupby("user_id")["item_id"]
    .apply(set)
    .to_dict()
)

def get_candidates_for_user_safe(user_id, n=CANDIDATES_PER_USER):
    candidates = []
    
    if user_id in user2idx:
        uidx = user2idx[user_id]
        if uidx < sparse_user_item.shape[0]:
            user_vec = sparse_user_item[uidx]
            try:
                rec_idxs, rec_scores = als_model.recommend(
                    uidx, user_vec, N=n // 2, filter_already_liked_items=True
                )
                for idx, score in zip(rec_idxs, rec_scores):
                    if idx in idx2item:
                        candidates.append((idx2item[idx], score, "als"))
            except Exception as e:
                pass 
    
   
    if user_id in user_seen:
        seen_items = list(user_seen[user_id])
        recent_items = seen_items[-5:]  
        seen_set = set(seen_items)
        
        for seed_item in recent_items:
            if seed_item not in item2idx:
                continue
            seed_iidx = item2idx[seed_item]
            if seed_iidx >= als_model.item_factors.shape[0]:
                continue
            
            seed_vec = als_model.item_factors[seed_iidx].reshape(1, -1)
            sims = als_model.item_factors @ seed_vec.T
            sims = sims.flatten()
            top_idxs = np.argsort(sims)[::-1][1:20]
            
            for idx in top_idxs:
                if idx >= len(idx2item):
                    continue
                cand_item = idx2item[idx]
                if cand_item not in seen_set:
                    candidates.append((cand_item, float(sims[idx]), "knn"))
    
    seen_set = user_seen.get(user_id, set())
    cand_set = {c[0] for c in candidates}
    pop_count = 0
    for pop_item in popularity_fallback:
        if pop_item not in seen_set and pop_item not in cand_set:
            candidates.append((pop_item, 0.0, "pop"))
            pop_count += 1
            if pop_count >= n // 4:
                break
    
    seen_cands = set()
    unique_candidates = []
    for c in candidates:
        if c[0] not in seen_cands:
            seen_cands.add(c[0])
            unique_candidates.append(c)
    
    return unique_candidates[:n]

lgb_rows = []
lgb_labels = []
lgb_groups = []
TRAIN_CANDS_PER_USER = 50

for i, uid in enumerate(tqdm(val_users_in_train, desc="Построение признаков")):
    candidates = get_candidates_for_user_safe(uid, n=TRAIN_CANDS_PER_USER)
    if not candidates:
        continue
    
    relevant = val_relevant.get(uid, set())
    tag_p, auth_p, ser_p = build_user_profile(uid, train_agg)
    
    group_size = 0
    for item_id, score, source in candidates:
        label = 2 if (item_id in relevant) else 0
        
        if "is_purchased" in val_data.columns:
            purchased = val_data[
                (val_data["user_id"] == uid) & 
                (val_data["item_id"] == item_id) & 
                (val_data["is_purchased"] == True)
            ]
            if len(purchased) > 0:
                label = 3
        
        row = compute_features(uid, item_id, score, source, tag_p, auth_p, ser_p)
        lgb_rows.append(row)
        lgb_labels.append(label)
        group_size += 1
    
    lgb_groups.append(group_size)

X_train = pd.DataFrame(lgb_rows)
y_train = np.array(lgb_labels)
groups_train = np.array(lgb_groups)

print(f"Размер обучающей выборки: {X_train.shape}")
print(f"Доля позитивных примеров: {(y_train > 0).mean():.3f}")

Построение обучающих данных для LightGBM...
Пользователей для обучения LightGBM: 5000


Построение признаков: 100%|██████████| 5000/5000 [19:56<00:00,  4.18it/s]


Размер обучающей выборки: (102543, 20)
Доля позитивных примеров: 0.017


In [29]:
# Обучение LightGBM ранкера
print("Обучение LightGBM ранкера")
lgb_train = lgb.Dataset(X_train, label=y_train, group=groups_train)

params = {
    "objective": "lambdarank",
    "metric": "ndcg",
    "ndcg_eval_at": [20],
    "learning_rate": 0.05,
    "num_leaves": 63,
    "min_data_in_leaf": 5,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "verbose": -1,
    "n_jobs": -1,
    "label_gain": [0, 1, 3, 7],
}

lgb_model = lgb.train(
    params,
    lgb_train,
    num_boost_round=200,
    valid_sets=[lgb_train],
    callbacks=[lgb.log_evaluation(50)],
)

print("Модель обучена!")
print("\nВажность признаков (топ-10):")
feature_importance = pd.Series(
    lgb_model.feature_importance("gain"),
    index=X_train.columns
).sort_values(ascending=False)
print(feature_importance.head(10))

Обучение LightGBM ранкера...
[50]	training's ndcg@20: 0.947889
[100]	training's ndcg@20: 0.959741
[150]	training's ndcg@20: 0.968199
[200]	training's ndcg@20: 0.970567
Модель обучена!

Важность признаков (топ-10):
user_interactions     11905.363947
user_purchase_rate    11183.133105
item_interactions      6016.195984
item_pop_rank          4437.687788
item_avg_rating        4340.102367
user_unique_items      2458.687333
source_pop             2108.412970
item_unique_users       553.843079
als_score                88.398494
source_knn                1.988230
dtype: float64


In [35]:
# Переобучение iALS на полных данных

print("Переобучение iALS на полных данных")

full_agg = compute_weights(train)
full_agg["uidx"] = full_agg["user_id"].map(user2idx)
full_agg["iidx"] = full_agg["item_id"].map(item2idx)

new_users = full_agg["user_id"][full_agg["uidx"].isna()].unique()
new_items = full_agg["item_id"][full_agg["iidx"].isna()].unique()

for u in new_users:
    user2idx[u] = len(user2idx)
for it in new_items:
    idx = len(item2idx)
    item2idx[it] = idx
    idx2item[idx] = it

full_agg["uidx"] = full_agg["user_id"].map(user2idx)
full_agg["iidx"] = full_agg["item_id"].map(item2idx)

n_users_full = len(user2idx)
n_items_full = len(item2idx)

print(f"Полных пользователей: {n_users_full:,}")
print(f"Полных товаров: {n_items_full:,}")

sparse_iu_full = csr_matrix(
    (full_agg["weight"].values * ALS_ALPHA,
     (full_agg["iidx"].values, full_agg["uidx"].values)),
    shape=(n_items_full, n_users_full)
)

als_model_full = als.AlternatingLeastSquares(
    factors=128, iterations=25, regularization=0.01,
    random_state=RANDOM_SEED, use_gpu=False,
)
als_model_full.fit(sparse_iu_full)
sparse_ui_full = sparse_iu_full.T.tocsr()

user_seen_full = full_agg.groupby("user_id")["item_id"].apply(set).to_dict()

print("Модель на полных данных обучена")

Переобучение iALS на полных данных...
Полных пользователей: 348,111
Полных товаров: 31,221


  0%|          | 0/25 [00:00<?, ?it/s]

Модель на полных данных обучена!


In [44]:
# Генерация финальных рекомендаций
print("Генерация финальных рекомендаций")

print("Кеширование ALS рекомендаций")
als_recs_cache = {}
for user_id in tqdm(test_users["user_id"].values, desc="ALS cache"):
    if user_id in user2idx:
        uidx = user2idx[user_id]
        if uidx < sparse_ui_full.shape[0]:
            user_vec = sparse_ui_full[uidx]
            try:
                rec_idxs, rec_scores = als_model_full.recommend(
                    uidx, user_vec, N=100, filter_already_liked_items=True
                )
                als_recs_cache[user_id] = [
                    (idx2item[idx], float(score)) 
                    for idx, score in zip(rec_idxs, rec_scores) 
                    if idx in idx2item
                ]
            except:
                als_recs_cache[user_id] = []
        else:
            als_recs_cache[user_id] = []
    else:
        als_recs_cache[user_id] = []

print("Кеширование профилей пользователей")
user_profiles_cache = {}
for user_id in tqdm(test_users["user_id"].values, desc="Profiles"):
    user_profiles_cache[user_id] = build_user_profile(user_id, full_agg)

item_factors_full = als_model_full.item_factors
n_items_full = item_factors_full.shape[0]

results = []
test_user_list = test_users["user_id"].tolist()
batch_size = 500

for start_idx in tqdm(range(0, len(test_user_list), batch_size), desc="Batches"):
    batch_users = test_user_list[start_idx:start_idx + batch_size]
    
    batch_data = []
    batch_seen = {}
    
    for user_id in batch_users:
        seen = user_seen_full.get(user_id, set())
        batch_seen[user_id] = seen

        for item_id, score in als_recs_cache.get(user_id, [])[:60]:
            if item_id not in seen:
                batch_data.append((user_id, item_id, score, "als"))

        seen_list = list(seen)[-5:]
        for seed_item in seen_list:
            if seed_item in item2idx:
                seed_iidx = item2idx[seed_item]
                if seed_iidx < n_items_full:
                    seed_vec = item_factors_full[seed_iidx]
                    sims = item_factors_full @ seed_vec
                    top_idxs = np.argsort(sims)[::-1][1:10]
                    for idx in top_idxs:
                        if idx < len(idx2item):
                            cand_item = idx2item[idx]
                            if cand_item not in seen:
                                batch_data.append((user_id, cand_item, float(sims[idx]), "knn"))
 
        pop_added = 0
        for pop_item in popularity_fallback:
            if pop_item not in seen:
                batch_data.append((user_id, pop_item, 0.0, "pop"))
                pop_added += 1
                if pop_added >= 20:
                    break
    
    if not batch_data:
        continue
    
    batch_unique = {}
    for uid, iid, score, source in batch_data:
        key = (uid, iid)
        if key not in batch_unique or score > batch_unique[key][0]:
            batch_unique[key] = (score, source)
    
    batch_features = []
    batch_items_info = []
    
    for (uid, iid), (score, source) in batch_unique.items():
        tag_p, auth_p, ser_p = user_profiles_cache.get(uid, ({}, {}, {}))
        feat = compute_features(uid, iid, score, source, tag_p, auth_p, ser_p)
        batch_features.append(feat)
        batch_items_info.append((uid, iid))
    
    X_batch = pd.DataFrame(batch_features)
    pred_scores = lgb_model.predict(X_batch)
    
    user_candidates = defaultdict(list)
    for (uid, iid), pred_score in zip(batch_items_info, pred_scores):
        user_candidates[uid].append((iid, pred_score))
    
    for uid, cands in user_candidates.items():
        cands.sort(key=lambda x: x[1], reverse=True)
        seen = batch_seen[uid]
        recs = []
        for iid, _ in cands:
            if iid not in seen and iid not in recs:
                recs.append(iid)
            if len(recs) >= TOP_K:
                break
        while len(recs) < TOP_K:
            for pop_item in popularity_fallback:
                if pop_item not in seen and pop_item not in recs:
                    recs.append(pop_item)
                if len(recs) >= TOP_K:
                    break
        for iid in recs[:TOP_K]:
            results.append({"user_id": uid, "item_id": iid})

print(f"Сгенерировано рекомендаций: {len(results):,}")

Генерация финальных рекомендаций...
Кеширование ALS рекомендаций...


ALS cache: 100%|██████████| 185282/185282 [02:36<00:00, 1183.11it/s] 


Кеширование профилей пользователей...


Batches: 100%|██████████| 371/371 [5:21:17<00:00, 51.96s/it]  

Сгенерировано рекомендаций: 3,705,640


In [49]:


del train, items, test_users

import gc
gc.collect()

1351

In [54]:
print("Сохранение сабмита")

submission = pd.DataFrame(results)
submission['user_id'] = submission['user_id'].astype('int32')
submission['item_id'] = submission['item_id'].astype('int32')

counts = submission.groupby("user_id").size()
print(f"Рекомендаций на пользователя — мин: {counts.min()}, макс: {counts.max()}, среднее: {counts.mean():.1f}")

submission.to_csv("submission_lgbm_2.csv", index=False)
print(f"Сохранено в submission_lgbm_2.csv")
print(f"Всего строк: {len(submission):,}")


Сохранение сабмита
Рекомендаций на пользователя — мин: 20, макс: 20, среднее: 20.0
Сохранено в submission_lgbm_2.csv
Всего строк: 3,705,640


In [55]:
from IPython.display import FileLink

display(FileLink('submission_lgbm_2.csv'))

/kaggle/working/submission_lgbm_2.csv

Score: 0.06124

In [1]:
# Ячейка 1: Загрузка данных
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
from tqdm import tqdm
import gc
import warnings
warnings.filterwarnings('ignore')

print("Загрузка данных...")
train = pd.read_parquet('/kaggle/input/datasets/vasilisadan18/hw4-recsys/train.pq')
test_users = pd.read_csv('/kaggle/input/datasets/vasilisadan18/hw4-recsys/test_users.csv')

train['user_id'] = train['user_id'].astype('int32')
train['item_id'] = train['item_id'].astype('int32')
train['is_purchased'] = train['is_purchased'].astype('int8')
train['timestamp'] = pd.to_datetime(train['timestamp'])
train = train.sort_values('timestamp')

print(f"Train: {len(train):,}, Test users: {len(test_users):,}")

Загрузка данных...
Train: 11,373,426, Test users: 185,282


In [2]:
# Ячейка 2: Создание матрицы покупок и обучение EASE
print("Создание матрицы покупок...")
purchases = train[train['is_purchased'] == 1]

# Топ-20 000 популярных товаров для ограничения памяти
item_counts = purchases['item_id'].value_counts()
top_items = item_counts.nlargest(20000).index
purchases_top = purchases[purchases['item_id'].isin(top_items)]

users = purchases_top['user_id'].unique()
items_list = top_items.tolist()

user2idx = {u: i for i, u in enumerate(users)}
item2idx = {it: i for i, it in enumerate(items_list)}
idx2item = {i: it for it, i in item2idx.items()}

rows = purchases_top['user_id'].map(user2idx).values
cols = purchases_top['item_id'].map(item2idx).values
data = np.ones(len(rows), dtype='float32')
X = csr_matrix((data, (rows, cols)), shape=(len(users), len(items_list)))

print(f"Матрица: {X.shape[0]:,} × {X.shape[1]:,}, non-zero: {X.nnz:,}")

print("Обучение EASE...")
lambda_reg = 500.0
G = X.T @ X
G_dense = G.toarray().astype(np.float32)
G_dense += np.eye(len(items_list), dtype=np.float32) * lambda_reg
B = np.linalg.solve(G_dense, G.toarray()).astype(np.float32)
np.fill_diagonal(B, 0)
print("EASE обучена!")

del G, G_dense, X, rows, cols, data
gc.collect()

Создание матрицы покупок...
Матрица: 303,082 × 20,000, non-zero: 4,437,312
Обучение EASE...
EASE обучена!


30

In [3]:
# Ячейка 3: Популярность и история
print("Популярность...")
popularity = train.groupby('item_id').size().reset_index(name='count')
popularity = popularity.sort_values('count', ascending=False)
top_popular = popularity.head(500)['item_id'].values

print("История пользователей...")
user_history = {}
for uid, group in tqdm(train.groupby('user_id'), desc="История"):
    purchased = group[group['is_purchased'] == 1]
    user_history[uid] = {
        'interacted': set(group['item_id'].values),
        'purchased': set(purchased['item_id'].values)
    }

del train
gc.collect()

Популярность...
История пользователей...


История: 100%|██████████| 348111/348111 [03:23<00:00, 1711.92it/s]


0

In [5]:
# Ячейка 4 (БЫСТРАЯ): Генерация рекомендаций через матричное умножение
print("Генерация рекомендаций (векторизованная)...")

# Создаём user-item матрицу для тестовых пользователей
test_user_ids = test_users['user_id'].values
n_test = len(test_user_ids)
n_items = len(items_list)

# Заполняем матрицу: 1 если пользователь покупал товар
test_matrix = np.zeros((n_test, n_items), dtype=np.float32)

for i, uid in enumerate(tqdm(test_user_ids, desc="Построение матрицы")):
    hist = user_history.get(uid, {'purchased': set()})
    for item in hist['purchased']:
        if item in item2idx:
            test_matrix[i, item2idx[item]] = 1.0

print(f"Матрица: {test_matrix.shape}")
print("Вычисление скоров...")

# Векторизованное предсказание: (n_test, n_items) = (n_test, n_items) @ (n_items, n_items)
scores_matrix = test_matrix @ B

print("Сортировка и выбор топ-20...")
results = []

for i, uid in enumerate(tqdm(test_user_ids, desc="Сортировка")):
    scores = scores_matrix[i]
    # Получаем топ-100 индексов
    top_indices = np.argpartition(scores, -100)[-100:]
    top_indices = top_indices[np.argsort(scores[top_indices])[::-1]]
    
    interacted = user_history.get(uid, {'interacted': set()})['interacted']
    
    recs = []
    for idx in top_indices:
        item = idx2item[idx]
        if item not in interacted:
            recs.append(item)
        if len(recs) >= 20:
            break
    
    # Добиваем популярными
    if len(recs) < 20:
        for item in top_popular:
            if item not in interacted and item not in recs:
                recs.append(item)
            if len(recs) >= 20:
                break
    
    for item in recs[:20]:
        results.append({'user_id': int(uid), 'item_id': int(item)})

print(f"Сгенерировано: {len(results):,}")

Генерация рекомендаций (векторизованная)...


Построение матрицы: 100%|██████████| 185282/185282 [00:03<00:00, 53139.77it/s]


Матрица: (185282, 20000)
Вычисление скоров...
Сортировка и выбор топ-20...


Сортировка: 100%|██████████| 185282/185282 [00:18<00:00, 9982.90it/s] 

Сгенерировано: 3,705,640


In [6]:
# Ячейка 5: Сохранение
print("Сохранение")
submission = pd.DataFrame(results)
submission['user_id'] = submission['user_id'].astype('int32')
submission['item_id'] = submission['item_id'].astype('int32')
submission.to_csv('submission_ease.csv', index=False)
print(f"Сохранено: {submission.shape}")

Сохранение
Сохранено: (3705640, 2)


In [7]:
from IPython.display import FileLink

display(FileLink('submission_ease.csv'))

/kaggle/working/submission_ease.csv

Score: 0.07126